# 2.3 Statistische Auswertung und Zufallszahlen

In den letzten beiden Kapiteln haben wir das Schwingungssignal unserer Maschine
modelliert und kalibriert. In der Realität sind Messsignale jedoch nie
perfekt: Jeder Sensor fügt dem Signal ein kleines Rauschen hinzu, das aus
elektronischem Schaltungsrauschen, Vibrationen der Umgebung oder
Temperaturschwankungen entsteht. Um zu verstehen, wie stark dieses Rauschen
das Messergebnis beeinflusst, brauchen wir statistische Kenngrößen wie
Mittelwert und Standardabweichung. NumPy stellt diese direkt als Funktionen
bereit und kann außerdem realistische Zufallssignale erzeugen.

## Lernziele

* [ ] Sie können mit `np.mean()`, `np.std()`, `np.min()` und `np.max()`
  statistische Kenngrößen eines Arrays berechnen.
* [ ] Sie können mit `np.random.normal()` und `np.random.uniform()`
  Zufallszahlen erzeugen.
* [ ] Sie können einem Signal synthetisches Rauschen hinzufügen.
* [ ] Sie können mit `np.random.seed()` Zufallsexperimente reproduzierbar
  machen.

## Statistische Kenngrößen

Wir starten direkt mit unserem Maschinensignal aus Kapitel 2.1 und berechnen
seine wichtigsten statistischen Kenngrößen:

In [ ]:
import numpy as np

# Gedämpftes Schwingungssignal (wie in Kapitel 2.1)
A     = 5.0    # Anfangsamplitude in m/s²
delta = 1.5    # Dämpfungskoeffizient in 1/s
f     = 10.0   # Frequenz in Hz
t     = np.linspace(0, 2, 1000) # Zeitache in s
a     = A * np.exp(-delta * t) * np.sin(2 * np.pi * f * t) # Schwingung in m/s²

# Statistische Kenngrößen
print(f"Mittelwert:         {np.mean(a):.4f} m/s^2")
print(f"Standardabweichung: {np.std(a):.4f} m/s^2")
print(f"Minimum:            {np.min(a):.4f} m/s^2")
print(f"Maximum:            {np.max(a):.4f} m/s^2")

Der Mittelwert liegt nahe bei null, was für ein Schwingungssignal sinnvoll
ist: Positive und negative Ausschläge heben sich gegenseitig auf. Die
Standardabweichung beschreibt, wie stark die Werte im Mittel vom Mittelwert
abweichen, also die typische Schwingungsamplitude im betrachteten Zeitfenster.

Neben diesen Funktionen sind auch `.sum()` für die Summe aller Elemente und
`np.argmax()` für den Index des größten Elements nützlich:

In [ ]:
idx_max = np.argmax(a)
print(f"Größter Wert bei t = {t[idx_max]:.4f} s")

`np.argmax()` gibt nicht den Wert selbst zurück, sondern seinen Index im
Array. Mit `t[idx_max]` lesen wir dann den zugehörigen Zeitpunkt ab.

### Mini-Übung 1

Wir betrachten die Beschleunigungswerte der drei Sensoren aus Kapitel 2.2:

```python
# Messwerte: 3 Sensoren, 4 Zeitpunkte (in m/s²)
messungen = np.array([
    [0.3, 1.2, 2.5, 1.8],   # Sensor 1
    [0.5, 0.9, 1.7, 1.1],   # Sensor 2
    [1.2, 2.4, 3.1, 2.0],   # Sensor 3
])
```

1. Berechnen Sie den Mittelwert und die Standardabweichung des gesamten Arrays.
2. Berechnen Sie den Mittelwert jedes Sensors separat, also den Mittelwert über
   jede Zeile. Hinweis: `np.mean(messungen, axis=1)` mittelt jeden Sensor über
   alle Zeitpunkte, d.h. entlang die Spalten (`axis=1`) einer Zeile. Das Ergebnis ist
   ein Wert pro Sensor.
3. Welcher Sensor hat die größte mittlere Beschleunigung?

In [ ]:
# Code-Zelle

## Zufallszahlen mit `np.random`

Um das Messrauschen unserer Sensoren zu simulieren, brauchen wir Zufallszahlen.
NumPy stellt dafür das Untermodul `np.random` bereit. Die zwei wichtigsten
Funktionen sind `np.random.normal()` für normalverteiltes Rauschen und
`np.random.uniform()` für gleichverteilte Zufallszahlen.

Normalverteiltes Rauschen ist in der Messtechnik am häufigsten anzutreffen,
weil viele unabhängige Störquellen zusammenwirken. Wir erzeugen es mit:

```python
np.random.normal(loc, scale, size)
```

`loc` ist der Erwartungswert (meist 0 für Rauschen), `scale` ist die
Standardabweichung und `size` die Anzahl der Zufallszahlen:

In [ ]:
# Rauschen mit Standardabweichung 0.2 m/s²
rauschen = np.random.normal(loc=0.0, scale=0.2, size=1000)

print(f"Mittelwert des Rauschens:         {np.mean(rauschen):.4f} m/s²")
print(f"Standardabweichung des Rauschens: {np.std(rauschen):.4f} m/s²")

Der Mittelwert liegt nahe bei null und die Standardabweichung nahe bei 0.2,
wie erwartet. Die leichten Abweichungen entstehen, weil es sich um endlich
viele Zufallszahlen handelt.

**Hinweis: Wie können Zufallszahlen reproduzierbar gemacht werden?**
Zufallszahlen in NumPy sind pseudozufällig: Sie werden aus einem
Startwert berechnet und sehen zufällig aus, sind aber deterministisch.
Mit `np.random.seed(42)` legen wir diesen Startwert fest. Jeder, der
denselben **Seed** verwendet, erhält exakt dieselbe Folge von Zufallszahlen.
Das ist wichtig für reproduzierbare Simulationen und Tests. Ohne
`seed()` wählt NumPy bei jedem Programmstart einen anderen Startwert,
und die Ergebnisse ändern sich bei jedem Durchlauf.

## Verrauschtes Messsignal

Jetzt fügen wir dem Schwingungssignal unserer Maschine realistisches
Messrauschen hinzu. Das entspricht genau dem, was ein echter Sensor liefern
würde:

In [ ]:
np.random.seed(42)

# Sauberes Signal (wie in Kapitel 2.1)
a_sauber = A * np.exp(-delta * t) * np.sin(2 * np.pi * f * t)

# Verrauschtes Signal: Rauschen addieren
sigma_rauschen = 0.3    # Standardabweichung des Rauschens in m/s²
rauschen       = np.random.normal(0.0, sigma_rauschen, size=len(t))
a_verrauscht   = a_sauber + rauschen

# Vergleich der Kenngrößen
print("Sauberes Signal:")
print(f"  Standardabweichung: {np.std(a_sauber):.4f} m/s^2")
print(f"  Maximum:            {np.max(a_sauber):.4f} m/s^2")

print("Verrauschtes Signal:")
print(f"  Standardabweichung: {np.std(a_verrauscht):.4f} m/s^2")
print(f"  Maximum:            {np.max(a_verrauscht):.4f} m/s^2")

Die Standardabweichung des verrauschten Signals ist größer als die des
sauberen Signals, weil das Rauschen zur Gesamtvarianz beiträgt. Das Maximum
des verrauschten Signals kann deutlich vom sauberen Maximalwert abweichen,
weil ein einzelner Rauschausreißer den Maximalwert verfälschen kann. Das ist
ein wichtiger Grund, in der Messtechnik nie nur einen Maximalwert zu
betrachten, sondern immer statistische Kenngrößen wie die
Standardabweichung.

### Mini-Übung 2

Wir untersuchen, wie stark das Rauschen den Mittelwert des Signals verfälscht.

1. Erzeugen Sie mit `np.random.seed(0)` ein verrauschtes Signal mit einer
   Standardabweichung von `sigma = 0.5` m/s² (verwenden Sie dieselben
   Parameter `A`, `delta`, `f`, `t` wie oben).
2. Berechnen Sie den Mittelwert des sauberen und des verrauschten Signals.
3. Berechnen Sie die Abweichung zwischen beiden Mittelwerten. Was stellen
   Sie fest?

In [ ]:
# Code-Zelle

## Zusammenfassung und Ausblick

Wir haben das Schwingungssignal unserer Maschine statistisch ausgewertet und
mit realistischem Messrauschen versehen. `np.mean()` und `np.std()` liefern
Mittelwert und Standardabweichung eines Arrays, `np.min()`, `np.max()` und
`np.argmax()` geben Extremwerte und deren Position zurück. Mit
`np.random.normal()` erzeugen wir normalverteiltes Rauschen, das wir dem
sauberen Signal einfach addieren. `np.random.seed()` macht Zufallsexperimente
reproduzierbar.

Damit haben wir die wichtigsten Werkzeuge von NumPy für die Arbeit mit
eindimensionalen und zweidimensionalen Arrays kennengelernt: Erzeugen, Rechnen,
Lineare Algebra, Statistik und Zufallszahlen. Im Übungskapitel wenden wir all
das auf neue Problemstellungen an.